# Experiment 4 — stepped learning rate and EMA

This experiment keeps the Experiment 2 winning architecture unchanged. It tests Adam with learning-rate drops from `1e-4` to `3e-5` at step 10,000 and `9e-6` at step 14,000. Validation, trajectory plots, and final generation use exponential-moving-average weights (`0.999`).

The target is Experiment 3's best validation loss: **0.123984**.

In [ ]:
import os
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
os.chdir(project_root)

import torch
import wandb

from datasets.mnist import MNISTSampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer, model_size_b

CONFIG_PATH = "configs/experiment_4.yaml"
cfg = load_config(CONFIG_PATH)
persistent_root = Path(
    os.getenv("DIFFUSION_DATA_ROOT", "/workspace-global/Diffusion-data")
)
data_root = persistent_root / "datasets" / "mnist"
checkpoints_dir = persistent_root / "checkpoints" / "experiment_4"
samples_dir = persistent_root / "samples" / "experiment_4"
wandb_dir = Path("/workspace/wandb")
for path in (data_root, checkpoints_dir, samples_dir, wandb_dir):
    path.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Experiment 4 is intended to run on a CUDA GPU")
torch.set_float32_matmul_precision("high")
torch.manual_seed(cfg["data"]["seed"])
print("device", device, torch.cuda.get_device_name(0))
print("target validation loss", cfg["source"]["target_val_loss"])

In [ ]:
def make_path(split: str) -> GaussianConditionalProbabilityPath:
    return GaussianConditionalProbabilityPath(
        p_data=MNISTSampler(
            root=str(data_root),
            split=split,
            val_size=cfg["data"]["val_size"],
            seed=cfg["data"]["seed"],
        ),
        p_simple_shape=[1, cfg["data"]["image_size"], cfg["data"]["image_size"]],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)


train_path = make_path("train")
val_path = make_path("val")
model = FlowModel.from_config(CONFIG_PATH).to(device)
trainer = FlowTrainer(path=train_path, model=model, val_path=val_path)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"model size: {model_size_b(model) / 1024**2:.2f} MiB")

In [ ]:
wandb.login()
train_cfg = cfg["training"]
target_val_loss = cfg["source"]["target_val_loss"]

with wandb.init(
    project="mnist-flow-matching",
    dir=str(wandb_dir),
    name="experiment-4-step-lr-ema",
    tags=["experiment-4", "step-lr", "ema"],
    config=cfg,
) as run:
    checkpoint_path = checkpoints_dir / f"{run.id}.pt"
    best_checkpoint_path = checkpoints_dir / f"{run.id}_best.pt"
    history = trainer.train(
        num_steps=train_cfg["num_steps"],
        device=device,
        lr=train_cfg["learning_rate"],
        optimizer_name=train_cfg["optimizer"],
        weight_decay=train_cfg["weight_decay"],
        max_grad_norm=train_cfg["max_grad_norm"],
        lr_milestones=train_cfg["lr_milestones"],
        lr_gamma=train_cfg["lr_gamma"],
        ema_decay=train_cfg["ema_decay"],
        batch_size=cfg["data"]["batch_size"],
        ckpt_path=checkpoint_path,
        best_ckpt_path=best_checkpoint_path,
        checkpoint_every=train_cfg["checkpoint_every"],
        val_every=train_cfg["val_every"],
        val_batches=train_cfg["val_batches"],
        plot_every=train_cfg["plot_every"],
        n_plot_images=cfg["sampling"]["num_samples"],
        n_plot_steps=20,
        samples_dir=samples_dir,
        show_plots=False,
        wandb_run=run,
    )
    best_train_loss = history["train"].min().item()
    best_val_loss = history["val"].min().item()
    run.summary["loss/train_best"] = best_train_loss
    run.summary["loss/val_best"] = best_val_loss
    run.summary["target/val_loss"] = target_val_loss
    run.summary["target/beaten"] = best_val_loss < target_val_loss
    run.summary["checkpoint"] = str(checkpoint_path)
    run.summary["best_checkpoint"] = str(best_checkpoint_path)
    run.summary["training_steps"] = train_cfg["num_steps"]

best_state = torch.load(best_checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(best_state["ema_model"] or best_state["model"])
model.eval()
print("best training loss", best_train_loss)
print("best EMA validation loss", best_val_loss)
print("beat Experiment 3 target", best_val_loss < target_val_loss)
print("best checkpoint step", best_state["step"])
print("best checkpoint", best_checkpoint_path)

In [ ]:
import torch.nn.functional as F
from torchvision.utils import save_image

from sampling.ode import EulerSimulator, FlowODE

num_samples = cfg["sampling"]["num_samples"]
ode_steps = cfg["sampling"]["ode_steps"]
generator = torch.Generator(device=device).manual_seed(cfg["data"]["seed"])
noise = torch.randn(num_samples, 1, 32, 32, generator=generator, device=device)
ts = torch.linspace(0, 1, ode_steps + 1, device=device).expand(num_samples, -1)
samples = EulerSimulator(FlowODE(model)).simulate(noise, ts).clamp(-1, 1).cpu()

generation_dir = samples_dir / "final"
individual_dir = generation_dir / "individual"
individual_dir.mkdir(parents=True, exist_ok=True)
torch.save(samples, generation_dir / "samples.pt")
display_samples = F.interpolate(samples, size=(256, 256), mode="nearest")
save_image(
    display_samples,
    generation_dir / "grid_5x5.png",
    nrow=5,
    normalize=True,
    value_range=(-1, 1),
    padding=4,
    pad_value=1,
)
for index, sample in enumerate(display_samples):
    save_image(
        sample,
        individual_dir / f"sample_{index + 1:02d}.png",
        normalize=True,
        value_range=(-1, 1),
    )

print("grid", generation_dir / "grid_5x5.png")